In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from numba import config as config_numba
from phenoscapes.feature_extraction import extract_features
from phenoscapes.montage import generate_overview_montage
from phenoscapes.sc import convert_to_h5ad, plot_summary
from skimage import io
from skimage.color import label2rgb
from skimage.transform import rescale
from tqdm import tqdm

np.random.seed(0)
config_numba.CPU_NAME = "generic"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

In [ ]:
# read anndata for ecm
adata = ad.read_h5ad("early_brain_organoid_4i_ecm.h5ad")
color_volume_midnight_blue = mpl.colors.LinearSegmentedColormap.from_list(
    "", ["#c9c7c7", "#191970"]
)
color_pallete_perturbation = {"Matrigel": "#17ad97", "No Matrix": "#4d4d4d"}

In [ ]:
brain_regions_colors = {
    "N. Epi.": "#CC9933",
    "NCCs": "#F4A261",
    "NC neurons": "#f70a69",
    "Non-Tel. neurons": "#930740",
    "Die. Prog.": "#b36bff",
    "Pros. Prog.": "#506E8A",
    "Tel. Prog.": "#049983",
    "Non-Tel. Prog.": "#adbdff",
    "Unknown": "#9e9e9e",
}

cross_tab = pd.crosstab(
    adata.obs["leiden"], adata.obs["Cluster_annotations"], normalize="columns"
).T
cross_tab = cross_tab.reindex(
    [
        "Unknown",
        "N. Epi.",
        "Pros. Prog.",
        "NC neurons",
        "NCCs",
        "Non-Tel. Prog.",
        "Non-Tel. neurons",
        "Die. Prog.",
        "Tel. Prog.",
    ]
)
colors = ["#A3CEBD", "#3293B3", "#F4ADC7", "#2F7A5F", "#FF8500", "#F92174", "#9CDEE1"]
cluster_colors = ListedColormap(colors)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

tmp = cross_tab.plot(kind="bar", stacked=True, cmap=cluster_colors)
tmp.legend(title="ECM cluster", bbox_to_anchor=(1.5, 1.02), loc="upper right")
tmp.grid(False)
tmp.spines["top"].set_visible(False)
tmp.spines["right"].set_visible(False)
tmp.spines["bottom"].set_visible(False)
tmp.spines["left"].set_visible(False)


In [ ]:
hue_order = [
    "Unknown",
    "N. Epi.",
    "Pros. Prog.",
    "NC neurons",
    "NCCs",
    "Non-Tel. Prog.",
    "Non-Tel. neurons",
    "Die. Prog.",
    "Tel. Prog.",
]
proteins = [
    "HAPLN1",
    "LAMA1",
    "COL2A1",
    "COL4A1",
    "FN1",
    "IGFBP2",
    "VCAN",
]

In [ ]:
gray_turqoise_midnight_blue = mpl.colors.LinearSegmentedColormap.from_list(
    "", ["#f2f2f2", "#8afbff", "#191970"]
)
with plt.rc_context():
    sc.pl.dotplot(
        adata,
        proteins,
        groupby="leiden",
        cmap=gray_turqoise_midnight_blue,
        dendrogram=False,
        standard_scale="var",
        show=False,
        return_fig=False,
    )

In [ ]:
with plt.rc_context():
    sc.pl.dotplot(
        adata,
        proteins,
        groupby="Cluster_annotations",
        categories_order=hue_order,
        cmap=gray_turqoise_midnight_blue,
        dendrogram=False,
        standard_scale="var",
        show=False,
        return_fig=False,
    )

In [ ]:
sc.set_figure_params(dpi=200, vector_friendly=False)

sc.pl.umap(
    adata,
    color="Condition",
    size=15,
    title="",
    frameon=False,
    legend_fontsize="x-small",
    palette=color_pallete_perturbation,
)

In [ ]:
sc.set_figure_params(dpi=200, vector_friendly=False)

sc.pl.umap(
    adata,
    color="Day",
    size=15,
    title="",
    frameon=False,
    legend_fontsize="x-small",
    palette="tab20",
)

In [ ]:
sc.set_figure_params(dpi=200, vector_friendly=False)

with plt.rc_context():
    sc.pl.umap(
        adata,
        color="leiden",
        size=15,
        title="",
        frameon=False,
        legend_fontsize="x-small",
        palette=colors,
        show=False,
    )

In [ ]:
sc.set_figure_params(dpi=200, vector_friendly=False)

with plt.rc_context():
    sc.pl.umap(
        adata,
        color="Cluster_annotations",
        size=15,
        title="",
        frameon=False,
        legend_fontsize="x-small",
        palette=brain_regions_colors,
        show=False,
    )

In [ ]:
def to_shape(a, shape):
    y_, x_ = shape
    y, x = a.shape
    y_pad = y_ - y
    x_pad = x_ - x
    return np.pad(
        a,
        ((y_pad // 2, y_pad // 2 + y_pad % 2), (x_pad // 2, x_pad // 2 + x_pad % 2)),
        mode="constant",
    )


samples_montage_clusters = np.unique(adata.obs["sample"])
dir_segmented = Path(dir_output, "segmented_cell_nuclei")
dir_segmented_cell = Path(dir_output, "segmented_cytoplasma")
dir_segmented_ecm = Path(dir_output, "segmented_ecm_niche")
# colors = sns.color_palette('husl', len(np.unique(adata.obs['leiden'])))
colors = sns.color_palette(colors)
obs_name = "leiden"
slice_step = 2000
shape = 500
downscale = 0.25
imgs_full = []


mask_shapes = []
for sample in tqdm(samples_montage_clusters):
    adata_well = adata[adata.obs["sample"] == sample].copy()
    mask = io.imread(Path(dir_segmented, sample + ".tif"))
    mask = rescale(mask, downscale, order=0, preserve_range=True, anti_aliasing=False)
    mask_shapes.append(max(mask.shape))
max_shape = max(mask_shapes)

for sample in tqdm(samples_montage_clusters):
    adata_well = adata[adata.obs["sample"] == sample].copy()

    mask = io.imread(Path(dir_segmented_ecm, sample + ".tif"))

    mask = rescale(mask, downscale, order=0, preserve_range=True, anti_aliasing=False)
    mask = to_shape(mask, (max_shape, max_shape))
    mask_colored = np.zeros(mask.shape).astype(np.float32)
    for i in np.unique(adata_well.obs["ID"]):
        adata_i = adata_well[adata_well.obs["ID"] == i]
        mask_colored[mask == i] = (
            1 + np.array(adata_i.obs[obs_name]).astype(np.float32)[0]
        )
    colors_2 = []
    for cluster_num in (np.unique(mask_colored)[1:] - 1).astype(int):
        colors_2.append(colors[cluster_num])
    labels_rgb = label2rgb(mask_colored, bg_label=0, colors=colors_2)
    dpi = mpl.rcParams["figure.dpi"]
    fig = plt.figure(figsize=(mask_colored.shape[1] / dpi, mask_colored.shape[0] / dpi))
    fig.tight_layout()
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(labels_rgb)
    ax.axis("off")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)
    plt.axis("off")
    canvas = plt.gca().figure.canvas
    canvas.draw()
    data = np.frombuffer(canvas.tostring_rgb(), dtype=np.uint8)
    image = data.reshape(canvas.get_width_height()[::-1] + (3,))